## tl;dr

**READY_FOR_V3_RESEARCH** across 10 pairs, 20 files, and 165,034 candles.

- Validity failures: 0.
- 4h-to-1d reconciliation mismatches: 0.
- Minimum declared-split coverage: 100.0000%.
- Largest source gap: 1 candles.


## Context & Methods

This notebook validates Binance Spot 4h and 1d OHLCV for the ten-pair V3 USDT universe.

### Key assumptions

- Grain is one UTC exchange candle per pair, timeframe, and candle-open timestamp.
- Source is Binance public OHLCV downloaded with Freqtrade 2026.7 / CCXT 4.5.68.
- The research windows are train `2021-01-01/2024-01-01`, validation `2024-01-01/2026-01-01`, and holdout `2026-01-01/2026-08-01`.
- The incomplete current day is outside the declared holdout.


## Data

### 1. Load the fixed universe

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "user_data" / "data" / "binance"
REPORT_DIR = PROJECT_ROOT / "user_data" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ("BTC", "ETH", "BNB", "SOL", "XRP", "ADA", "DOGE", "LINK", "LTC", "AVAX")
INTERVALS = {"4h": pd.Timedelta(hours=4), "1d": pd.Timedelta(days=1)}
frames = {}
for base in PAIRS:
    for timeframe in INTERVALS:
        path = DATA_DIR / f"{base}_USDT-{timeframe}.feather"
        frame = pd.read_feather(path).sort_values("date").reset_index(drop=True)
        frame["date"] = pd.to_datetime(frame["date"], utc=True)
        frames[(base, timeframe)] = frame

len(frames), sum(len(frame) for frame in frames.values())

(20, 165034)

## Results

### 2. Validate grain, domains, and gaps

In [2]:
profile_rows = []
gap_rows = []

for (base, timeframe), frame in frames.items():
    interval = INTERVALS[timeframe]
    dates = frame["date"].drop_duplicates().sort_values().reset_index(drop=True)
    expected = int((dates.iloc[-1] - dates.iloc[0]) / interval) + 1
    deltas = dates.diff()
    for position in deltas[deltas > interval].index:
        gap_rows.append({
            "pair": f"{base}/USDT",
            "timeframe": timeframe,
            "previous_utc": dates.iloc[position - 1],
            "next_utc": dates.iloc[position],
            "missing_candles": int(deltas.iloc[position] / interval) - 1,
        })

    invalid_ohlc = (
        (frame["high"] < frame[["open", "close", "low"]].max(axis=1))
        | (frame["low"] > frame[["open", "close", "high"]].min(axis=1))
    )
    profile_rows.append({
        "pair": f"{base}/USDT",
        "timeframe": timeframe,
        "rows": len(frame),
        "from_utc": dates.iloc[0],
        "to_utc": dates.iloc[-1],
        "coverage_since_listing_pct": 100 * len(dates) / expected,
        "duplicate_timestamps": int(frame["date"].duplicated().sum()),
        "null_cells": int(frame.isna().sum().sum()),
        "nonpositive_prices": int((frame[["open", "high", "low", "close"]] <= 0).any(axis=1).sum()),
        "negative_volume": int((frame["volume"] < 0).sum()),
        "invalid_ohlc": int(invalid_ohlc.sum()),
    })

profile_df = pd.DataFrame(profile_rows)
gaps_df = pd.DataFrame(gap_rows)
profile_df.to_csv(REPORT_DIR / "v3_data_quality_profile.csv", index=False)
gaps_df.to_csv(REPORT_DIR / "v3_data_quality_gaps.csv", index=False)
display(profile_df)
display(gaps_df.sort_values("missing_candles", ascending=False).head(20) if not gaps_df.empty else gaps_df)

,pair,timeframe,rows,from_utc,to_utc,coverage_since_listing_pct,duplicate_timestamps,null_cells,nonpositive_prices,negative_volume,invalid_ohlc
0,BTC/USDT,4h,14439,2020-01-01 00:00:00+00:00,2026-08-03 12:00:00+00:00,99.993075,0,0,0,0,0
1,BTC/USDT,1d,2406,2020-01-01 00:00:00+00:00,2026-08-02 00:00:00+00:00,100.000000,0,0,0,0,0
2,ETH/USDT,4h,14439,2020-01-01 00:00:00+00:00,2026-08-03 12:00:00+00:00,99.993075,0,0,0,0,0
3,ETH/USDT,1d,2406,2020-01-01 00:00:00+00:00,2026-08-02 00:00:00+00:00,100.000000,0,0,0,0,0
4,BNB/USDT,4h,14439,2020-01-01 00:00:00+00:00,2026-08-03 12:00:00+00:00,99.993075,0,0,0,0,0
5,BNB/USDT,1d,2406,2020-01-01 00:00:00+00:00,2026-08-02 00:00:00+00:00,100.000000,0,0,0,0,0
6,SOL/USDT,4h,13101,2020-08-11 04:00:00+00:00,2026-08-03 12:00:00+00:00,100.000000,0,0,0,0,0
7,SOL/USDT,1d,2183,2020-08-11 00:00:00+00:00,2026-08-02 00:00:00+00:00,100.000000,0,0,0,0,0
8,XRP/USDT,4h,14439,2020-01-01 00:00:00+00:00,2026-08-03 12:00:00+00:00,99.993075,0,0,0,0,0
9,XRP/USDT,1d,2406,2020-01-01 00:00:00+00:00,2026-08-02 00:00:00+00:00,100.000000,0,0,0,0,0


,pair,timeframe,previous_utc,next_utc,missing_candles
0,BTC/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
1,ETH/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
2,BNB/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
3,XRP/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
4,ADA/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
5,DOGE/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
6,LINK/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1
7,LTC/USDT,4h,2020-02-19 08:00:00+00:00,2020-02-19 16:00:00+00:00,1


### 3. Verify all declared chronological splits

In [3]:
SPLITS = {
    "train": (pd.Timestamp("2021-01-01", tz="UTC"), pd.Timestamp("2024-01-01", tz="UTC")),
    "validation": (pd.Timestamp("2024-01-01", tz="UTC"), pd.Timestamp("2026-01-01", tz="UTC")),
    "holdout": (pd.Timestamp("2026-01-01", tz="UTC"), pd.Timestamp("2026-08-01", tz="UTC")),
}

split_rows = []
for split, (start, end) in SPLITS.items():
    for (base, timeframe), frame in frames.items():
        interval = INTERVALS[timeframe]
        selected = frame[(frame["date"] >= start) & (frame["date"] < end)]
        expected = int((end - start) / interval)
        unique = selected["date"].nunique()
        split_rows.append({
            "split": split,
            "pair": f"{base}/USDT",
            "timeframe": timeframe,
            "rows": len(selected),
            "expected_rows": expected,
            "missing_candles": expected - unique,
            "coverage_pct": 100 * unique / expected,
        })

split_df = pd.DataFrame(split_rows)
split_df.to_csv(REPORT_DIR / "v3_data_quality_splits.csv", index=False)
display(split_df)
display(split_df.groupby(["split", "timeframe"])["coverage_pct"].agg(["min", "mean", "max"]))

,split,pair,timeframe,rows,expected_rows,missing_candles,coverage_pct
0,train,BTC/USDT,4h,6570,6570,0,100.0
1,train,BTC/USDT,1d,1095,1095,0,100.0
2,train,ETH/USDT,4h,6570,6570,0,100.0
3,train,ETH/USDT,1d,1095,1095,0,100.0
4,train,BNB/USDT,4h,6570,6570,0,100.0
5,train,BNB/USDT,1d,1095,1095,0,100.0
6,train,SOL/USDT,4h,6570,6570,0,100.0
7,train,SOL/USDT,1d,1095,1095,0,100.0
8,train,XRP/USDT,4h,6570,6570,0,100.0
9,train,XRP/USDT,1d,1095,1095,0,100.0


min   mean    max
split      timeframe                     
holdout    1d         100.0  100.0  100.0
           4h         100.0  100.0  100.0
train      1d         100.0  100.0  100.0
           4h         100.0  100.0  100.0
validation 1d         100.0  100.0  100.0
           4h         100.0  100.0  100.0

### 4. Reconcile complete 4h days with stored daily candles

In [4]:
consistency_rows = []
for base in PAIRS:
    four_hour = frames[(base, "4h")].set_index("date").copy()
    four_hour["_count"] = 1
    aggregated = four_hour.resample("1d", label="left", closed="left").agg({
        "open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum", "_count": "sum"
    })
    aggregated = aggregated[aggregated["_count"] == 6].drop(columns="_count")
    daily = frames[(base, "1d")].set_index("date")[["open", "high", "low", "close", "volume"]]
    joined = aggregated.join(daily, how="inner", lsuffix="_4h", rsuffix="_1d")
    row = {"pair": f"{base}/USDT", "complete_days_compared": len(joined)}
    for column in ("open", "high", "low", "close", "volume"):
        matches = np.isclose(
            joined[f"{column}_4h"], joined[f"{column}_1d"], rtol=1e-10, atol=1e-10
        )
        row[f"{column}_mismatches"] = int((~matches).sum())
    consistency_rows.append(row)

consistency_df = pd.DataFrame(consistency_rows)
consistency_df.to_csv(REPORT_DIR / "v3_data_quality_timeframe_consistency.csv", index=False)
display(consistency_df)

C:\Users\dioni\AppData\Local\Temp\ipykernel_9152\1148889421.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  aggregated = four_hour.resample("1d", label="left", closed="left").agg({
C:\Users\dioni\AppData\Local\Temp\ipykernel_9152\1148889421.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  aggregated = four_hour.resample("1d", label="left", closed="left").agg({
C:\Users\dioni\AppData\Local\Temp\ipykernel_9152\1148889421.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  aggregated = four_hour.resample("1d", label="left", closed="left").agg({
C:\Users\dioni\AppData\Local\Temp\ipykernel_9152\1148889421.py:5: Pandas4Warning: 'd' is deprecated and will be removed in a future version, please use 'D' instead.
  aggregated = four_hour.resample("1d", label="left", closed="left").agg({
C:\Users\dioni\AppData\Local\Temp\ipyker

,pair,complete_days_compared,open_mismatches,high_mismatches,low_mismatches,close_mismatches,volume_mismatches
0,BTC/USDT,2405,0,0,0,0,0
1,ETH/USDT,2405,0,0,0,0,0
2,BNB/USDT,2405,0,0,0,0,0
3,SOL/USDT,2182,0,0,0,0,0
4,XRP/USDT,2405,0,0,0,0,0
5,ADA/USDT,2405,0,0,0,0,0
6,DOGE/USDT,2405,0,0,0,0,0
7,LINK/USDT,2405,0,0,0,0,0
8,LTC/USDT,2405,0,0,0,0,0
9,AVAX/USDT,2140,0,0,0,0,0


### 5. Produce the data-quality decision

In [5]:
validity_columns = [
    "duplicate_timestamps", "null_cells", "nonpositive_prices", "negative_volume", "invalid_ohlc"
]
validity_failures = int(profile_df[validity_columns].sum().sum())
mismatch_columns = [column for column in consistency_df if column.endswith("_mismatches")]
timeframe_mismatches = int(consistency_df[mismatch_columns].sum().sum())
minimum_split_coverage = float(split_df["coverage_pct"].min())
largest_gap = int(gaps_df["missing_candles"].max()) if not gaps_df.empty else 0

decision = {
    "decision": "READY_FOR_V3_RESEARCH" if (
        validity_failures == 0 and timeframe_mismatches == 0 and minimum_split_coverage >= 99.5
    ) else "NEEDS_REVISION",
    "pairs": len(PAIRS),
    "files": len(frames),
    "candles": int(sum(len(frame) for frame in frames.values())),
    "validity_failures": validity_failures,
    "timeframe_mismatches": timeframe_mismatches,
    "minimum_split_coverage_pct": minimum_split_coverage,
    "largest_gap_candles": largest_gap,
    "as_of_utc": max(frame["date"].max() for frame in frames.values()).isoformat(),
}
(REPORT_DIR / "v3_data_quality_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
decision

{'decision': 'READY_FOR_V3_RESEARCH',
 'pairs': 10,
 'files': 20,
 'candles': 165034,
 'validity_failures': 0,
 'timeframe_mismatches': 0,
 'minimum_split_coverage_pct': 100.0,
 'largest_gap_candles': 1,
 'as_of_utc': '2026-08-03T12:00:00+00:00'}

## Takeaways

- All ten pairs cover the complete train, validation, and sealed holdout windows at a minimum of 100.0000%.
- Stored daily candles reconcile with complete 4-hour aggregates and the OHLC domains are valid.
- The data is suitable for the declared V3 research, subject to the documented current-universe survivorship caveat.
